In [0]:
# Databricks notebook source
# =============================================================================
# SILVER LAYER : d_Customer Transformation (UNITY CATALOG PROD STANDARD)
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, TimestampType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging

# Run these at the top of your notebook
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoOptimize.autoCompact.enabled", "true")

# Set up logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_d_customer")

print("⚪ INITIALIZING PROD SILVER PIPELINE : CUSTOMER ENTITY")

# =============================================================================
# 1. CONFIGURATION (STRICT UNITY CATALOG)
# =============================================================================
CATALOG_NAME = "prx"
BRONZE_SCHEMA = "prx_bronze"
SILVER_SCHEMA = "prx_silver"

# Fully Qualified Table Names
BRONZE_TABLE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.bronze_account"
SILVER_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.customer"
EXCEPTION_TABLE = f"{CATALOG_NAME}.{SILVER_SCHEMA}.central_exceptions"

# External Locations mapped earlier
STORAGE_ACCOUNT = "stpraxaslakehouse"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/customer"
EXCEPTION_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/central_exceptions"

MERGE_KEYS = ["CustSrcId"]

# Ensure Silver Schema Exists Securely
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}")

# Capture the true executing user for Auditing
executing_user = spark.sql("SELECT current_user()").collect()[0][0]

# =============================================================================
# 2. UTILITIES
# =============================================================================
def safe_str(c): 
    return F.coalesce(F.col(c).cast(StringType()), F.lit(""))

def empty_str_to_null_ts(c):
    return F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), None)\
            .otherwise(F.col(c)).cast(TimestampType())

# =============================================================================
# 3. READ & BASE TRANSFORMATION
# =============================================================================
logger.info("Reading Bronze Data from Unity Catalog...")
bronze_df = spark.table(BRONZE_TABLE).filter(F.col("IsSales").cast(IntegerType()) == 1)

logger.info("Applying Business Transformations & Audit Logs...")
silver_df = bronze_df.select(
    # Core Keys
    F.col("Id").cast(StringType()).alias("CustSrcId"),
    F.coalesce(F.trim(F.col("Code")), F.lit("")).alias("CustKey"),
    
    # Contact & Location Info
    safe_str("Name").alias("CustName"),
    safe_str("City").alias("City"),
    safe_str("State").alias("State"),
    safe_str("Country").alias("Country"),
    safe_str("PostCode").alias("ZipCd"),
    safe_str("Phone").alias("TelPhNo"),
    safe_str("Email").alias("EmailAddr"),
    
    # Financial & Sales attributes
    safe_str("PaymentConditionSales").alias("TermsCd"),
    safe_str("AccountManager").alias("AcctMngr"),
    safe_str("VATNumber").alias("VATNo"),
    safe_str("SalesCurrency").alias("SalesCurr"),
    safe_str("MainContact").alias("CustContactSrcId"),
    safe_str("GLAccountPurchase").alias("GLPurchAcctSrcId"),
    safe_str("GLAccountSales").alias("GLSalesAcctSrcId"),
    
    # Hardcoded Enterprise Values
    F.lit("T1").alias("CustType"),
    F.lit("Tire-1").alias("CustTypeDesc"),
    F.lit("Active").alias("CustStatus"),

    # Timestamps
    empty_str_to_null_ts("Created").alias("CreatedDt"),
    empty_str_to_null_ts("Modified").alias("UpdatedDt"),

    # Enterprise Audit Tracking
    F.current_timestamp().alias("SysCreatedDt"),
    F.current_timestamp().alias("SysUpdatedDt"),
    F.lit(executing_user).alias("SysUpdatedBy")
)

# =============================================================================
# 4. DATA QUALITY & DEDUPLICATION RULES
# =============================================================================
logger.info("Evaluating Data Quality Rules...")

# Rule 1: Missing ID Check
dq_df = silver_df.withColumn(
    "DQ_Reason",
    F.array_remove(F.array(
        F.when(F.col("CustSrcId") == "", F.lit("MISSING_ID"))
    ), None)
).withColumn(
    "DQ_Status",
    F.when(F.size("DQ_Reason") > 0, F.lit(2)).otherwise(F.lit(0))
)

# Rule 2: Deduplication Check
window_dup = Window.partitionBy("CustSrcId").orderBy(F.col("CustSrcId").desc_nulls_last())

dq_df = dq_df.withColumn(
    "rn", F.row_number().over(window_dup)
).withColumn(
    "DQ_Status",
    F.when(F.col("rn") > 1, F.lit(4)).otherwise(F.col("DQ_Status"))
).drop("rn")

# ⚡ STAFF ENGINEER FIX: Cache the dataframe before running counts to save DAG re-computes!
dq_df.cache()

valid_df = dq_df.filter(F.col("DQ_Status") == 0).drop("DQ_Reason", "DQ_Status")
error_df = dq_df.filter(F.col("DQ_Status").isin(2, 4))

valid_count = valid_df.count()
error_count = error_df.count()

logger.info(f"✅ Valid Records: {valid_count:,}")
logger.info(f"❌ Error Records: {error_count:,}")

# =============================================================================
# 5. CENTRALIZED EXCEPTION ROUTING
# =============================================================================
if error_count > 0:
    logger.info("Routing exceptions to Central Quarantine Unity Catalog Table...")
    exception_df = error_df.select(
        F.lit(SILVER_TABLE).alias("table_name"),
        F.col("CustSrcId").alias("record_key"), 
        F.when(F.col("DQ_Status") == 4, F.lit("Duplicate Record"))
         .otherwise(
            F.concat(F.lit("Validation Failed: "), F.concat_ws(", ", F.col("DQ_Reason")))
         ).alias("exception_details"),
        F.current_timestamp().alias("syscreateddt")
    )

    if spark.catalog.tableExists(EXCEPTION_TABLE):
        exception_df.write.format("delta").mode("append").saveAsTable(EXCEPTION_TABLE)
    else:
        exception_df.write.format("delta").mode("overwrite").option("path", EXCEPTION_PATH).saveAsTable(EXCEPTION_TABLE)

# =============================================================================
# 6. MERGE TO SILVER DELTA (UPSERT)
# =============================================================================
logger.info("Executing MERGE INTO Silver Delta Table...")

if valid_count > 0:
    # Final Dedup Safety
    window_spec = Window.partitionBy(*MERGE_KEYS).orderBy(F.col("UpdatedDt").desc_nulls_last())
    final_df = valid_df.withColumn("rn", F.row_number().over(window_spec)).filter("rn = 1").drop("rn")

    if spark.catalog.tableExists(SILVER_TABLE):
        delta_tbl = DeltaTable.forName(spark, SILVER_TABLE)
        
        # Dynamic Merge Condition Builder
        cond = " AND ".join([f"tgt.{k} = src.{k}" for k in MERGE_KEYS])
        
        (delta_tbl.alias("tgt")
         .merge(final_df.alias("src"), cond)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        final_df.write.format("delta").mode("overwrite").option("path", SILVER_PATH).saveAsTable(SILVER_TABLE)

# Free memory after operations
dq_df.unpersist()

print("🎉 PROD SILVER PIPELINE COMPLETED SUCCESSFULLY!")

# COMMAND ----------

# MAGIC %sql 
# MAGIC -- Unity Catalog Verification
# MAGIC SELECT * FROM prx.prx_silver.customer LIMIT 10;

In [0]:
%sql
select * from prx.prx_silver.customer limit 10;